# Cisco LLM Analyzer and Meta-Analyzer experiment (RQ1-C extension)

Runs Cisco's LLM Analyzer (B) and Meta-Analyzer (C-det, C) over **every skill of a finished notebook-01 scan run**:
the same sample manifest, the same exported files, and the same baseline reports (copied and SHA-256 checked, never
re-run or modified). The work happens in `src/gitskills_cisco_llm`, so the same notebook handles 30, 100, 300 or
1,000+ skills. Only the settings below change.

| label | Cisco flags (plus `--format json --policy balanced`) |
|---|---|
| A_static / A_behavioral | baseline from notebook 01 (reused) |
| B_llm | `--use-behavioral --use-llm --verbose` |
| Cdet_meta | `--use-behavioral --enable-meta --verbose` |
| C_llm_meta | `--use-behavioral --use-llm --enable-meta --verbose` |

**Safety and cost.** Skill content is sent to the configured LLM. A local `ollama/...` model costs nothing. A hosted
model is refused unless `SKILL_SCANNER_LLM_API_KEY` is set in the environment (never written to disk) **and**
`BUDGET_USD` is set. Scanning stops before a scan would push the total past the budget. Every attempt, including failed
and interrupted ones, is logged in `attempts.jsonl` with tokens and cost.

**Resume.** Re-running the *Run* cell skips every finished scan (checkpoint `reports/<config>.run.json` with matching
fingerprint and report hash), so no finished (paid) request is repeated. A scan that was in flight when the kernel died
is re-run and shows up in `attempts_interrupted`.

Facts about Cisco 2.3.1 used here: `docs/cisco_llm_meta_analyzers.md`.

## 1. Settings

In [ ]:
SOURCE_RUN = "results/cisco_dataset_runs/agent_skills_sample/runs/2026-11-03_173516_sample30-seed2027"  # notebook-01 *scan* run
RUN_DIR = "results/cisco_dataset_runs/agent_skills_sample/runs/2026-11-07_222052_sample30-seed2027-llm-meta-qwen3-4b"
# RUN_DIR = None -> prepare a new run folder from SOURCE_RUN with the settings below (an existing RUN_DIR is resumed as is)

EXPERIMENT_NAME = "sample30-seed2027-llm-meta-qwen3-4b"
MODEL = "ollama/skillscan-qwen3-4b-ctx64k"   # local, free. Paid example: "claude-sonnet-5-5" (Cisco's default model)
MODES = ["behavioral_llm", "behavioral_meta", "behavioral_llm_meta"]
BUDGET_USD = None          # REQUIRED for a paid model: the approved spending cap in US dollars
WORKERS = 1                # parallel Cisco processes; keep 1 for local Ollama, 2-4 for a hosted API (rate limits)
MAX_ATTEMPTS = 3           # per scan, for transient errors (rate limit, timeout, connection); Cisco retries each call 3x too

## 2. Prepare (or reopen) the run folder

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "notebooks").is_dir() and (p / "src").is_dir())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
from gitskills_cisco_llm import Experiment, LLMSettings, Runner, prepare
from gitskills_cisco_llm.analyze import analyze, estimate_costs

if RUN_DIR is None:
    exp = Experiment(name=EXPERIMENT_NAME, modes=MODES, llm=LLMSettings(model=MODEL), workers=WORKERS,
                     budget_usd=BUDGET_USD, max_attempts=MAX_ATTEMPTS)
    RUN_DIR = prepare(ROOT, SOURCE_RUN, exp)
RUN_DIR = Path(RUN_DIR)
plan = pd.read_csv(RUN_DIR / "plan.csv")
display(plan.groupby(["configuration", "action"]).size().unstack(fill_value=0))

## 3. Run (resumable; safe to interrupt and re-run)

In [ ]:
runner = Runner(ROOT, RUN_DIR)
state = runner.execute()
print("state:", state)
status = json.loads((RUN_DIR / "execution_status.json").read_text())
{k: status[k] for k in ("attempts_started", "attempts_finished", "attempts_interrupted", "spent_usd", "pending", "stop_reason")}

## 4. Analyse and rebuild the report (`results/reports/...`)

In [ ]:
tables = analyze(ROOT, RUN_DIR)
for name in ("coverage_by_configuration", "findings_by_configuration", "meta_decision_summary", "runtime_and_cost"):
    print(name); display(tables[name])

## 5. What would the same experiment cost on a hosted model? (no API call)

In [ ]:
display(estimate_costs(ROOT, RUN_DIR, "claude-sonnet-5-5"))

## Scaling to 100 / 300 / 1,000 skills

1. `notebooks/03_stratified_sample.ipynb`: same `SEED = 2027`, larger `STRATUM_SIZES`. The draw takes the smallest
   random keys per stratum, so a larger sample **contains** the smaller one.
2. `notebooks/01_cisco_scanner_dataset_ready.ipynb`: `RUN_MODE = "scan"`, `SELECTION_MODE = "manifest"`,
   `SAMPLE_MANIFEST` = the new manifest. This exports the folders and runs the deterministic baseline (~6 s per call).
3. This notebook with `SOURCE_RUN` = that run and `RUN_DIR = None`.

Command-line equivalent (from the repository root):

```bash
PYTHONPATH=src .venv/bin/python -m gitskills_cisco_llm prepare --source <notebook-01 scan run> --name <name> --model <model> [--budget-usd X]
PYTHONPATH=src .venv/bin/python -m gitskills_cisco_llm run <run folder>
PYTHONPATH=src .venv/bin/python -m gitskills_cisco_llm analyze <run folder>
```